# AgroGuard Model Training Pipeline
This notebook provides the complete code to load, preprocess, and train the MobileNetV2 models for Crop Growth Stage Classification (9 classes, Weeks 1-9) and Disease Detection using PyTorch.

### Step 1: Import Dependencies

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from PIL import Image
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

### Step 2: Define Data Transforms & Loaders

In [ ]:
DATA_DIR = "/content/drive/MyDrive/crop_data"

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Assuming folders: DATA_DIR/train and DATA_DIR/val
# train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=train_transforms)
# val_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'val'), transform=val_transforms)

# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
# val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

### Step 3: Initialize MobileNetV2 Model

In [ ]:
NUM_CLASSES = 9  # For Weeks 1-9 classification

model = models.mobilenet_v2(weights=models.MobileNetV2_Weights.DEFAULT)
# Freeze feature extraction layers (optional)
for param in model.features.parameters():
    param.requires_grad = False

# Modify the classifier layer
model.classifier[1] = nn.Linear(model.last_channel, NUM_CLASSES)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(model)

### Step 4: Define Loss and Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.001)

### Step 5: Training Loop

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        
    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {epoch_loss:.4f}")

### Step 6: Save Model Weights

In [ ]:
# Save state dict
torch.save(model.state_dict(), "/content/growth_stage_model.pt")

# Trigger browser download in Google Colab
from google.colab import files
files.download("/content/growth_stage_model.pt")